# CNN Gerçek Dünya Örneği: Giysi Sınıflandırma (Fashion MNIST)

## Senaryo

Bir **e-ticaret şirketinde** (Trendyol, H&M gibi) çalıştığını düşün.  
Kullanıcılar her gün binlerce yeni ürün yüklüyor. Bunları **otomatik olarak doğru kategoriye** koymak gerekiyor.

**Sorun:** Her ürünü elle kategorize etmek çok zaman alır.  
**Çözüm:** CNN ile ürün fotoğrafına bakarak kategoriyi otomatik belirle.

## Veri Seti: Fashion MNIST

Zalando (Avrupa'nın büyük e-ticaret şirketi) tarafından hazırlandı.

- **70.000 gri tonlamalı resim** (28×28 piksel)
- **10 giysi kategorisi**: T-shirt, Pantolon, Kazak, Elbise, Mont, Sandalet, Gömlek, Spor Ayakkabı, Çanta, Bot

## Neden CNN?

Bir giysinin **şeklini, dokusunu, kenarlarını** tanımak gerekiyor.  
FNN piksel konumlarını kaybeder, CNN ise görüntüdeki **uzamsal desenleri** korur ve öğrenir.


In [ ]:
# Gerekli paketleri kur (ilk çalıştırmada birkaç saniye sürebilir)
!pip install seaborn -q
print("Kurulum tamamlandı.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Kategori isimleri (Türkçe)
CLASS_NAMES = [
    'T-Shirt', 'Pantolon', 'Kazak', 'Elbise', 'Mont',
    'Sandalet', 'Gömlek', 'Spor Ayakkabı', 'Çanta', 'Bot'
]

print(f"TensorFlow: {tf.__version__}")
print(f"Kategoriler: {CLASS_NAMES}")

# --- nn3d: agi tarayicida canli 3D izlemek icin ---------------------------
import sys, pathlib
if not any(pathlib.Path(p, "nn3d").is_dir() for p in sys.path):
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "src"))
import nn3d


## Adım 1: Veriyi Yükle


In [ ]:
# Fashion MNIST TensorFlow'a dahil — internet bağlantısıyla otomatik indirilir
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

print("=== Veri Boyutları ===")
print(f"Eğitim resimleri: {X_train.shape}  → 60.000 resim, 28×28 piksel")
print(f"Test resimleri:   {X_test.shape}   → 10.000 resim")
print(f"Piksel değer aralığı: {X_train.min()} - {X_train.max()} (0-255 gri tonlama)")

In [ ]:
# Her kategoriden bir örnek göster
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i, ax in enumerate(axes.flat):
    # Her sınıftan ilk örneği bul
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[i], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Fashion MNIST — Her Kategoriden Bir Örnek', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Normalizasyon ve boyut ekleme

# 1. Normalizasyon: 0-255 → 0.0-1.0
X_train = X_train / 255.0
X_test = X_test / 255.0

# 2. CNN için kanal boyutu ekle
# Fashion MNIST gri tonlamalı → 1 kanal
# CIFAR-10 renkli → 3 kanal (RGB)
# (60000, 28, 28) → (60000, 28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Eğitim verisi şekli: {X_train.shape}")
print("  → (örnek sayısı, yükseklik, genişlik, kanal sayısı)")

## Adım 2: Gelişmiş CNN Mimarisi

Bu sefer önceki basit CNN'den daha gelişmiş bir mimari kullanıyoruz:

### BatchNormalization Nedir?

Her katman çıkışını normalize eder (ortalama≈0, std≈1 yapar).  
Eğitimi hızlandırır, daha stabil yapar. Dropout ile birlikte güçlü bir kombinasyon.

### Conv Blok Mantığı:

```
Conv2D → BatchNorm → ReLU → MaxPool
```

Bu yapı birçok CNN mimarisinde tekrar eder (VGG, ResNet'in özü budur).


In [ ]:
# Gelişmiş CNN Mimarisi
model = models.Sequential([
    
    # === BLOK 1: Temel özellik çıkarımı ===
    # Düşük seviye özellikler: kenarlar, köşeler, gri tonlamalar
    layers.Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 1)),
    # padding='same': çıkış boyutunu giriş ile aynı tut (kenar kaybı yok)
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),        # 28×28 → 14×14
    layers.Dropout(0.25),
    
    # === BLOK 2: Orta seviye özellikler ===
    # Doku, kıvrım, desen örüntüleri
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),        # 14×14 → 7×7
    layers.Dropout(0.25),
    
    # === BLOK 3: Yüksek seviye özellikler ===
    # Şekil, siluet, nesne parçaları
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.25),
    
    # === SINIFLANDIRICI ===
    layers.Flatten(),                   # 7×7×128 = 6272 → düzleştir
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),               # Son dense katmanda daha güçlü dropout
    
    layers.Dense(10, activation='softmax')  # 10 kategori
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Adım 3: Eğitim — Early Stopping ile Akıllı Durdurma

Gerçek projelerde **kaç epoch eğiteceğimizi** baştan bilemeyiz.  
**EarlyStopping** callback'i bunu otomatik çözer:

- Validation loss iyileşmeyi durdurunca eğitimi kes
- `patience=5`: 5 epoch boyunca iyileşme olmazsa dur
- `restore_best_weights=True`: En iyi ağırlıklara geri dön


## Canlı 3D görselleştirmeBu ağda 16 katman var — nn3d hepsini soldan sağa dizer ve aralarındakibağlantıları çizer:- `Conv2D` katmanları **kanal** başına bir nokta gösterir (28×28×32 → 32 nokta).  6272 pikseli tek tek çizmek hem okunmaz hem tarayıcıyı kilitler.- `BatchNorm`, `Activation`, `MaxPool`, `Dropout` gibi ağırlıksız katmanlar  düz çizgilerle bağlanır — girişi aynen geçirdikleri için.- `Flatten` sonrası `Dense(256)` katmanında gerçek ağırlık matrisi görünür.Eğitim uzun sürüyor; sekmeyi açık bırakıp filtrelerin nasıl şekillendiğiniizleyebilirsin.

In [ ]:
# EarlyStopping: validation loss iyileşmezse dur
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau: öğrenme hızını dinamik olarak düşür
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,       # Öğrenme hızını yarıya indir
    patience=3,
    verbose=1
)

# nn3d: konvolüsyon katmanlarında her nokta bir ÖZELLİK HARİTASI (kanal),
# tek bir piksel değil. Kartta '32  28x28x32' yazar: 32 kanal, 28x28 uzamsal.
# Noktanın parlaklığı 'bu özellik haritası ne kadar aktif' demektir.
izleyici = nn3d.Monitor(
    X_test[:1],
    every=20,
    output_labels=CLASS_NAMES,
)

print("Eğitim başlıyor...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.1,    # Eğitim verisinin %10'unu validation olarak ayır
    callbacks=[early_stop, reduce_lr, izleyici],
    verbose=1
)

print("\nEğitim tamamlandı!")

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = len(history.history['accuracy'])

axes[0].plot(history.history['accuracy'], label='Eğitim', color='#9C27B0', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Doğrulama', color='#FF9800', 
             linestyle='--', linewidth=2)
axes[0].set_title(f'Doğruluk ({epochs_ran} Epoch)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Doğruluk')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Eğitim', color='#9C27B0', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Doğrulama', color='#FF9800',
             linestyle='--', linewidth=2)
axes[1].set_title('Kayıp')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kayıp')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN — Fashion MNIST Eğitim Geçmişi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Test seti değerlendirmesi
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Doğruluğu: %{test_acc*100:.2f}")
print(f"Test Kaybı: {test_loss:.4f}")

# Sınıf bazlı rapor
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("\n=== Sınıf Bazlı Performans ===")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix — hangi kategoriler karıştırılıyor?
cm = confusion_matrix(y_test, y_pred)
cm_percent = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(12, 10))
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix (Yüzde)', fontsize=14, fontweight='bold')
plt.ylabel('Gerçek Kategori')
plt.xlabel('Tahmin Edilen Kategori')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nDikkat edilecek noktalar:")
print("- T-Shirt ve Gömlek genellikle karıştırılır (her ikisi de düz giysi)")
print("- Kazak ve Mont benzer şekilli olduğundan karışabilir")
print("- Çanta ve Ayakkabı türleri genellikle net ayrılır")

In [ ]:
# Tahminleri görselleştir — doğru ve yanlışları göster
y_pred_probs = model.predict(X_test, verbose=0)

# Rastgele 12 test örneği seç
np.random.seed(42)
indices = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i, (idx, ax) in enumerate(zip(indices, axes.flat)):
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    
    pred = np.argmax(y_pred_probs[idx])
    confidence = y_pred_probs[idx][pred] * 100
    true_label = y_test[idx]
    
    color = '#4CAF50' if pred == true_label else '#F44336'
    
    ax.set_title(
        f"Tahmin: {CLASS_NAMES[pred]} (%{confidence:.0f})\nGerçek: {CLASS_NAMES[true_label]}",
        color=color, fontsize=9
    )
    ax.axis('off')

plt.suptitle('CNN Tahminleri (Yeşil=Doğru, Kırmızı=Yanlış)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Adım 4: CNN'in Ne Gördüğünü Anlamak — Feature Map Görselleştirme

CNN'in "aklından geçenleri" görmek için ara katman çıkışlarını görselleştirelim.  
Bu hem debugging için hem de CNN'in nasıl çalıştığını anlamak için çok değerlidir.


In [ ]:
# Bir test resmini seç
sample_idx = np.where(y_test == 0)[0][0]   # İlk T-Shirt örneği
sample_image = X_test[sample_idx:sample_idx+1]  # (1, 28, 28, 1)

# İlk Conv2D katmanının çıkışını al
# NOT: model.layers[0] Keras'ın otomatik eklediği InputLayer'ı döndürür.
# Bu yüzden indeks yerine tip kontrolüyle Conv2D katmanını buluyoruz.
conv_layers = [l for l in model.layers if isinstance(l, layers.Conv2D)]
first_conv = conv_layers[0]   # İlk Conv2D — 32 filtre

feature_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=first_conv.output
)

feature_maps = feature_extractor.predict(sample_image, verbose=0)
# feature_maps şekli: (1, 28, 28, 32) → 32 filtre

# İlk 16 filtreyi göster
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.set_title(f'Filtre {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle(
    f'İlk Conv2D Katmanının 16 Feature Map\'i\n(Giriş: {CLASS_NAMES[y_test[sample_idx]]})',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

# Orijinal resmi de göster
plt.figure(figsize=(3, 3))
plt.imshow(X_test[sample_idx].reshape(28, 28), cmap='gray')
plt.title(f'Orijinal: {CLASS_NAMES[y_test[sample_idx]]}')
plt.axis('off')
plt.show()

print("Her filtre resmin farklı bir özelliğine odaklanır:")
print("- Bazı filtreler yatay kenarları") 
print("- Bazıları dikey kenarları")
print("- Bazıları belirli dokuları yakalar")

## Sonuç

Bu notebook'ta:

1. **Fashion MNIST** gerçek e-ticaret senaryosuyla bağdaştırdık
2. **BatchNormalization + Dropout** kombinasyonuyla güçlü bir CNN inşa ettik
3. **EarlyStopping + ReduceLROnPlateau** ile akıllı eğitim uyguladık
4. **Confusion matrix** ile hangi kategorilerin karıştırıldığını gördük
5. **Feature map** görselleştirmesiyle CNN'in "içini" keşfettik

### Gerçek Hayatta Ne Olur?

E-ticaret şirketleri bu modeli:

- Yeni ürün yükleme sistemine entegre eder
- Satıcı ürün yüklediğinde otomatik kategori önerir
- Yanlış kategorideki ürünleri tespit eder
- Benzer ürün öneri sistemine girdi sağlar

### Geliştirme Fikirleri

- **Transfer Learning**: ResNet veya EfficientNet kullanarak %93+ doğruluk
- **Data Augmentation**: Resimleri döndürme, çevirme ile eğitim setini artırma
- **TensorFlow Serving**: Modeli REST API olarak sunma
